# SI26 — Week 6 Patch: Boost MIX Examples

### Fixes the "MIX F1 always 0.000" problem for Week 7

**Why this exists:** the original Week 6 dataset only has **12 MIX-labelled words** in
3182 total — so the Week 7 test split ends up with just **1 MIX example**, and no model
can score above 0.000 F1 on a single example unless it happens to guess that exact one
right. This notebook regenerates `dataset.csv` from the same source, but deliberately
pulls in more sentences containing common nativized English loanwords (`phone`,
`internet`, `mobile`, `message`, `order`, `email`, `link`, `wifi`, ...) so MIX has real
support to train and evaluate on.

Run this, then re-run the Week 7 notebook from Step 1 — it will automatically pick up
the new dataset (same GitHub/HF repo name, just richer in MIX examples).

In [1]:
import pandas as pd
import re
import random


## Step 1 — Download the same source data (unchanged)

In [2]:
url = "https://raw.githubusercontent.com/Smat26/Roman-Urdu-Dataset/master/Dataset/Roman%20Urdu%20DataSet.csv"
df = pd.read_csv(url, header=None, names=['sentence', 'sentiment', 'extra'],
                  on_bad_lines='skip', engine='python')

df = df.dropna(subset=['sentence'])
df['sentence'] = df['sentence'].astype(str).str.strip()
df = df[df.sentence.str.len() > 0].drop_duplicates(subset='sentence')
print(f"Total real sentences downloaded: {len(df)}")


Total real sentences downloaded: 19588


## Step 2 — Wider word lists (this is the actual fix)

In [3]:
urdu_markers = {'hai','hain','ka','ki','ke','tha','thi','nahi','nhi','mera','meri','bhai','yaar',
    'bohot','bahut','kya','kar','raha','rahi','hoon','hun','mein','main','se','ko','par','aur',
    'lekin','sab','bhi','kal','aaj','ye','wo','koi','kuch','acha','achi','bura','buri','tu','tum',
    'aap','hum','unko','uska','uski','iska','iski','sy','nai','hy','ni','wala','wali','bht'}

english_words = {'the','is','was','and','but','so','right','wrong','love','best','nice','good',
 'bad','sad','happy','sorry','please','thanks','thank','you','feel','feeling','time','life','world',
 'true','false','real','fake','proud','miss','missing','care','support','respect','trust','hope',
 'work','job','busy','tired','excited','stressed','boring','amazing','awesome','great','perfect',
 'beautiful','free','vote','election','change','future','past','present','history','story','song',
 'music','movie','actor','actress','player','team','match','game','sport','fan','fans','follow',
 'like','comment','share','post','video','photo','online','internet','phone','mobile','message',
 'text','call','email','plan','order','check','update','problem','issue','result','success','still'}

# EXPANDED — real nativized English loanwords used as everyday Urdu vocabulary.
# The original list only had 10 of these; widening it (and prioritizing MIX over ENG
# in label_word below) is what actually gives the model MIX examples to learn from.
mixwords = {'phone','internet','mobile','message','order','plan','time','net','system','job',
    'email','link','wifi','signal','battery','charger','screen','app','game','chat','call',
    'set','file','group','number','office','company','service','account','password','post',
    'share','video','photo','online','update','check','text','network','data','recharge',
    'balance','load','sim','card','ticket','seat','bus','van','rickshaw','taxi','hospital',
    'doctor','medicine','test','report','result','office','staff','manager','boss','duty',
    'shift','salary','bill','meter','light','fan','ac','fridge','tv','remote','button',
    'switch','table','chair','room','building','floor','lift','gate','road','traffic',
    'signal','petrol','diesel','tank','tyre','engine','brake','clutch','gear','license'}


## Step 3 — Filter code-switched sentences, oversampling MIX-word sentences

In [4]:
def is_code_switched(text):
    words = re.findall(r"[A-Za-z']+", text.lower())
    if not (5 <= len(words) <= 25):
        return False
    has_urdu = sum(1 for w in words if w in urdu_markers) >= 2
    has_eng  = sum(1 for w in words if w in english_words) >= 1
    return has_urdu and has_eng

def contains_mixword(text):
    words = re.findall(r"[A-Za-z']+", text.lower())
    return any(w in mixwords for w in words)

filtered = df[df.sentence.apply(is_code_switched)].sentence.tolist()
print(f"Total code-switched candidates: {len(filtered)}")

mix_candidates = [s for s in filtered if contains_mixword(s)]
other_candidates = [s for s in filtered if not contains_mixword(s)]
print(f"Candidates containing a MIX-list word: {len(mix_candidates)}")

random.seed(42)
random.shuffle(mix_candidates)
random.shuffle(other_candidates)

# Take ALL available MIX-word sentences (up to 150), then fill the rest with
# regular sentences up to a total sample size of 300 (bigger than Week 6's 200,
# which also helps overall training).
TARGET_TOTAL = 300
mix_sample = mix_candidates[:150]
remaining_slots = max(TARGET_TOTAL - len(mix_sample), 0)
other_sample = other_candidates[:remaining_slots]

sample = mix_sample + other_sample
random.shuffle(sample)
print(f"MIX-word sentences included: {len(mix_sample)}")
print(f"Other sentences included: {len(other_sample)}")
print(f"Total sampled: {len(sample)} sentences")


Total code-switched candidates: 1773
Candidates containing a MIX-list word: 322
MIX-word sentences included: 150
Other sentences included: 150
Total sampled: 300 sentences


## Step 4 — Word-level labelling (MIX checked before ENG)

In [5]:
def label_word(w):
    wl = re.sub(r'[^a-zA-Z]', '', w).lower()
    if not wl:
        return 'URD'
    if wl in mixwords:
        return 'MIX'
    if wl in english_words:
        return 'ENG'
    return 'URD'   # default: corpus is majority Roman Urdu

data = []
for s in sample:
    words = s.split()
    labels = [label_word(w) for w in words]
    data.append({'sentence': s, 'words': words, 'labels': labels})

print(f"Total labelled sentences: {len(data)}")


Total labelled sentences: 300


In [6]:
rows = []
for entry in data:
    for word, label in zip(entry['words'], entry['labels']):
        rows.append({
            'sentence': entry['sentence'],
            'word': word,
            'label': label
        })

df_final = pd.DataFrame(rows)
df_final.to_csv('dataset.csv', index=False, encoding='utf-8')

print(f'Dataset created: {len(df_final)} word entries')
print(f'Sentences: {df_final.sentence.nunique()}')
print('Label distribution:')
print(df_final.label.value_counts())
print(f"\nMIX went from 12 -> {(df_final.label == 'MIX').sum()} words ")
print('Skim df_final for any obviously wrong auto-labels before re-uploading.')


Dataset created: 4816 word entries
Sentences: 300
Label distribution:
label
URD    4367
ENG     271
MIX     178
Name: count, dtype: int64

MIX went from 12 -> 178 words 
Skim df_final for any obviously wrong auto-labels before re-uploading.


## Step 5 — Re-publish dataset (overwrites the same HF dataset repo)

In [7]:
from huggingface_hub import notebook_login
notebook_login()  # paste your HuggingFace WRITE token


In [8]:
from huggingface_hub import HfApi

HF_USERNAME = 'qandeelasim13'   # <-- edit if different
DATASET_REPO_NAME = 'code-switching-codesaviours-si26-qandeel'

api = HfApi()
api.upload_file(
    path_or_fileobj='dataset.csv',
    path_in_repo='dataset.csv',
    repo_id=f'{HF_USERNAME}/{DATASET_REPO_NAME}',
    repo_type='dataset',
)
print(f'Updated dataset.csv pushed to https://huggingface.co/datasets/{HF_USERNAME}/{DATASET_REPO_NAME}')
print('\nAlso replace dataset.csv in your GitHub repo (download it from this Colab session')
print('and re-upload/commit it) so the GitHub fallback in Week 7 stays in sync too.')


Updated dataset.csv pushed to https://huggingface.co/datasets/qandeelasim13/code-switching-codesaviours-si26-qandeel

Also replace dataset.csv in your GitHub repo (download it from this Colab session
and re-upload/commit it) so the GitHub fallback in Week 7 stays in sync too.


## Next step
Go back to `SI26-Week7-Qandeel.ipynb`, restart the runtime, and re-run from Step 1 —
it will now load this richer dataset (more MIX examples, more total sentences) and
MIX F1 should become a real, meaningful number instead of being capped at 0.000 by a
single test example.